In [7]:
from pathlib import Path
import pandas as pd

csv_path = Path("../data/raw/archive/HI-Small_Trans.csv")

columns = [
    "timestamp",
    "sender_bank",
    "sender_account",
    "receiver_bank",
    "receiver_account",
    "amount_received",
    "receiving_currency",
    "amount_paid",
    "payment_currency",
    "payment_format",
    "is_laundering",
]

preview = pd.read_csv(
    csv_path,
    header=0,
    names=columns,
    nrows=1000,
    dtype={
        "sender_bank": "string",
        "sender_account": "string",
        "receiver_bank": "string",
        "receiver_account": "string",
    },
)

display(preview.head(10))

,timestamp,sender_bank,sender_account,receiver_bank,receiver_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,2022/09/01 00:20,010,8000EBD30,010,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,03208,8000F4580,001,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,03209,8000F4670,03209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,012,8000F5030,012,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,010,8000F5200,010,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0
5,2022/09/01 00:03,001,8000F5AD0,001,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,0
6,2022/09/01 00:08,001,8000EBAC0,001,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,0
7,2022/09/01 00:16,001,8000EC1E0,001,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,0
8,2022/09/01 00:26,012,8000EC280,002439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,0
9,2022/09/01 00:21,001,8000EDEC0,0211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,0


In [8]:
preview["payment_format"].value_counts(dropna=False)

payment_format
Reinvestment    720
Credit Card     114
Cheque           88
ACH              38
Cash             28
Wire             12
Name: count, dtype: int64

In [9]:
from collections import Counter

payment_counts = Counter()
first_timestamp = None
last_timestamp = None
total_rows = 0

reader = pd.read_csv(
    csv_path,
    header=0,
    names=columns,
    usecols=["timestamp", "payment_format"],
    chunksize=100_000,
)

for chunk in reader:
    timestamps = pd.to_datetime(
        chunk["timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise",
    )

    chunk_start = timestamps.min()
    chunk_end = timestamps.max()

    first_timestamp = (
        chunk_start if first_timestamp is None
        else min(first_timestamp, chunk_start)
    )
    last_timestamp = (
        chunk_end if last_timestamp is None
        else max(last_timestamp, chunk_end)
    )

    payment_counts.update(
        chunk["payment_format"].fillna("<missing>")
    )
    total_rows += len(chunk)

summary = (
    pd.Series(dict(payment_counts), name="transactions")
    .sort_values(ascending=False)
    .to_frame()
)
summary["percent"] = (
    100 * summary["transactions"] / total_rows
).round(2)

print(f"Total transactions: {total_rows:,}")
print(f"First timestamp: {first_timestamp}")
print(f"Last timestamp:  {last_timestamp}")
display(summary)

Total transactions: 5,078,345
First timestamp: 2022-09-01 00:00:00
Last timestamp:  2022-09-18 16:18:00


,transactions,percent
Cheque,1864331,36.71
Credit Card,1323324,26.06
ACH,600797,11.83
Cash,490891,9.67
Reinvestment,481056,9.47
Wire,171855,3.38
Bitcoin,146091,2.88
